In [1]:

import pandas as pd 
import numpy as np 
import plotly_express as px 
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go
import itertools
from typing import List, Any, Optional


In [2]:
df = pd.read_pickle('lead_cleaning.pkl')

In [3]:
def get_most_recent_funnel_status(df, group_col = 'company_name', date_col='created_date', status_col='lead_status'):
    recent_df = (df.sort_values(by=date_col).drop_duplicates(subset=[group_col], keep='last'))
    return recent_df[[group_col, status_col]].reset_index(drop=True)

latest_status_df = get_most_recent_funnel_status(df)

latest_status_df

,company_name,lead_status
0,Northstar Analytics,Lost
1,Evergreen Pharma,Nurturing
2,Atlas Media Co,Converted
3,Oakline Telecom,Converted
4,Clearwater Foods,Converted
5,Summit Retail Group,Nurturing
6,Greenfield Energy,Converted
7,Redwood Consulting,Lost
8,Westbridge Legal,New
9,Pioneer Logistics,New


In [4]:
viz_df = df.groupby('company_name').agg(
    marketing_activity=('created_date', 'count'),
    sales_activity=('first_contact_date', 'count'),
).reset_index()

viz_df

,company_name,marketing_activity,sales_activity
0,Acme Health Systems,6,6
1,Atlas Media Co,3,1
2,Blue Ridge Financial,7,6
3,Brightpath Education,11,8
4,Cedar Manufacturing,5,3
5,Clearwater Foods,8,6
6,Evergreen Pharma,12,9
7,Greenfield Energy,4,4
8,Harbor Insurance,6,6
9,Ironwood Construction,5,5


In [5]:
joined_df = pd.merge(viz_df, latest_status_df, on='company_name', how='inner')
joined_df

,company_name,marketing_activity,sales_activity,lead_status
0,Acme Health Systems,6,6,Qualified
1,Atlas Media Co,3,1,Converted
2,Blue Ridge Financial,7,6,Qualified
3,Brightpath Education,11,8,Contacted
4,Cedar Manufacturing,5,3,New
5,Clearwater Foods,8,6,Converted
6,Evergreen Pharma,12,9,Nurturing
7,Greenfield Energy,4,4,Converted
8,Harbor Insurance,6,6,Nurturing
9,Ironwood Construction,5,5,Nurturing


In [6]:
numeric_columns = joined_df.select_dtypes(include=['number']).columns

scaler = MinMaxScaler()

joined_df[numeric_columns] = scaler.fit_transform(joined_df[numeric_columns])

joined_df

,company_name,marketing_activity,sales_activity,lead_status
0,Acme Health Systems,0.454545,0.625,Qualified
1,Atlas Media Co,0.181818,0.000,Converted
2,Blue Ridge Financial,0.545455,0.625,Qualified
3,Brightpath Education,0.909091,0.875,Contacted
4,Cedar Manufacturing,0.363636,0.250,New
5,Clearwater Foods,0.636364,0.625,Converted
6,Evergreen Pharma,1.000000,1.000,Nurturing
7,Greenfield Energy,0.272727,0.375,Converted
8,Harbor Insurance,0.454545,0.625,Nurturing
9,Ironwood Construction,0.363636,0.500,Nurturing


In [7]:
fig = px.scatter(
    joined_df, 
    x='marketing_activity',
    y='sales_activity',
    color='lead_status',
    symbol='lead_status', 
    hover_name='company_name',
    title='Company Activity Matrix by Funnel Status',
    labels={
        'marketing_activity':'Marketing Activity', 
        'sales_activity': 'Sales Activity',
        'lead_status': 'Last Funnel Status'
    }
)

fig.update_layout(
    xaxis_gridcolor='rgba(200, 200, 200, 0.3)',
    yaxis_gridcolor='rgba(200, 200, 200, 0.3)',
    plot_bgcolor='white'
)

fig.show()

In [8]:
table_view_df = df.groupby('campaign').agg(
    total_leads = ('lead_id', 'count'), 
    contacted = ('lead_status', lambda x: (x=='Contacted').sum()),
    sql = ('lead_status', lambda x: (x== 'Qualified').sum()),
    closed_won = ('lead_status', lambda x: (x=='Converted').sum()),
    new = ('lead_status', lambda x: (x=='New').sum())
)

# List of column pairs: (new_rate_col, numerator_col, denominator_col)
rate_cols = [
    ('contact_rate', 'contacted', 'total_leads'),
    ('sql_rate', 'sql', 'total_leads'),
    ('win_rate', 'closed_won', 'total_leads')
]

# Calculate, round, and add '%' instantly using vectorized string formatting
for new_col, num, denom in rate_cols:
    table_view_df[new_col] = ((table_view_df[num] / table_view_df[denom]) * 100).round(2).astype(str) + '%'
    
table_view_df


,total_leads,contacted,sql,closed_won,new,contact_rate,sql_rate,win_rate
campaign,,,,,,,,
Abm - Enterprise,19,3,2,1,6,15.79%,10.53%,5.26%
Content Download,12,2,1,5,1,16.67%,8.33%,41.67%
Fall Webinar,8,0,2,1,2,0.0%,25.0%,12.5%
Free Trial,15,1,2,5,2,6.67%,13.33%,33.33%
Google Search,11,3,1,3,3,27.27%,9.09%,27.27%
Healthcare Outreach,11,3,1,3,2,27.27%,9.09%,27.27%
Linkedin Prospecting,12,3,0,5,2,25.0%,0.0%,41.67%
Partner Referral,14,4,1,2,2,28.57%,7.14%,14.29%
Product Demo,14,2,2,4,3,14.29%,14.29%,28.57%


In [9]:


# 1. Corrected bracket grouping to cleanly combine list elements
all_labels = (
    list(df['campaign'].unique()) + 
    list(df['is_assigned'].unique()) + 
    list(df['lead_status'].unique())
)
label_indices = {label: idx for idx, label in enumerate(all_labels)}

# 2. Consistent flow_x variable names and 'target_label' headers
flow_1 = df.groupby(['campaign', 'is_assigned']).size().reset_index(name='value')
flow_1.columns = ['source_label', 'target_label', 'value']

flow_2 = df.groupby(['is_assigned', 'lead_status']).size().reset_index(name='value')
flow_2.columns = ['source_label', 'target_label', 'value']

combined_flows = pd.concat([flow_1, flow_2], ignore_index=True)

# 3. Fixed spelling of 'source_label' and clean mapping
combined_flows['source'] = combined_flows['source_label'].map(label_indices)
combined_flows['target'] = combined_flows['target_label'].map(label_indices)

# 4. Create and show the figure
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=18,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=all_labels,
        color="teal"
    ),
    link=dict(
        source=combined_flows['source'],
        target=combined_flows['target'],
        value=combined_flows['value'],
        hovertemplate='Flow: %{source.label} ➔ %{target.label}<br>Volume: %{value}<extra></extra>'
    )
)])

fig.update_layout(title_text="Customer Journey Funnel Analysis", font_size=12)
fig.show()


In [10]:
sales_table_df = df.groupby('sales_rep').agg(
    leads = ('lead_id', 'count')
)

sales_table_df

,leads
sales_rep,
Alex Kim,24
Chris Brown,17
Jordan Lee,13
Morgan Patel,18
Taylor Smith,18
Unassigned,33


In [11]:
class RoundRobinDistributor:
    def __init__(self, agents: List[str]):
        if not agents:
            raise ValueError("The agents list cannot be empty.")
        self.agents = agents
        self._cycler = itertools.cycle(self.agents)

    def assign_lead(self, lead_id: Any) -> str:
        return next(self._cycler)
if __name__ == '__main__': 
    unique_agents = df['sales_rep'].dropna().unique().tolist()
    unique_agents = [agent for agent in unique_agents if agent != 'Unassigned']
    
    distributor = RoundRobinDistributor(agents=unique_agents)
    
    df['round_robin_assignment'] = df['sales_rep']
    is_unassigned = df['round_robin_assignment'] == 'Unassigned'
    df.loc[is_unassigned, 'round_robin_assignment'] = [
    distributor.assign_lead(lead) for lead in df.loc[is_unassigned, 'lead_id']
    ]
    
df

,lead_id,lead_first_name,lead_last_name,job_title,company_name,industry,company_size,city,state,country,...,employees,estimated_deal_value_usd,sales_rep,last_activity,marketing_opt_in,notes,lead_full_name,is_decision_maker,is_assigned,round_robin_assignment
0,L-10019,James,Moore,Founder,Silverstone Automotive,Automotive,501-1000,Columbus,OH,United Sates,...,3200,15000,Alex Kim,Phone Call,NaN,Downloaded Whitepaper,James Moore,True,Assigned,Alex Kim
1,L-10046,Andrew,Jackson,Sales Director,Lighthouse Biotech,Biotechnology,1-50,Cambridge,MA,United Sates,...,8500,40000,Chris Brown,Form Submission,NaN,Asked For Pricing,Andrew Jackson,True,Assigned,Chris Brown
2,L-10048,Christopher,Williams,Product Marketing Manager,Vertex Software,Technology,5000+,Austin,TX,United Sates,...,120,75000,Unassigned,Form Submission,NaN,Referral From Partner,Christopher Williams,False,Unassigned,Alex Kim
3,L-10090,Andrew,Brown,Data Analytics Manager,Westbridge Legal,Legal,1001-5000,Denver,CO,United Sates,...,45,75000,Jordan Lee,Demo Requested,Yes,Duplicate Suspected,Andrew Brown,False,Assigned,Jordan Lee
4,L-10005,Emily,Garcia,Revops Manager,Metro Hospitality,Hospitality,5000+,Miami,FL,United Sates,...,12,5000,Chris Brown,Phone Call,No,Referral From Partner,Emily Garcia,False,Assigned,Chris Brown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,L-10107,Stephanie,Perez,Sales Director,Cedar Manufacturing,Manufacturing,5000+,Detroit,MI,United Sates,...,120,5000,Unassigned,Form Submission,NaN,Downloaded Whitepaper,Stephanie Perez,True,Unassigned,Jordan Lee
119,L-10015,Daniel,Anderson,Founder,Vertex Software,Technology,1-50,Austin,TX,United Sates,...,12,15000,Taylor Smith,Form Submission,No,Requested Demo,Daniel Anderson,True,Assigned,Taylor Smith
120,L-10093,Amanda,Brown,Product Marketing Manager,Cedar Manufacturing,Manufacturing,51-200,Detroit,MI,United Sates,...,1500,75000,Taylor Smith,Demo Requested,No,Wrong Contact,Amanda Brown,False,Assigned,Taylor Smith
121,L-10052,Anthony,White,Head Of Growth,Acme Health Systems,Healthcare,5000+,San Francisco,CA,United Sates,...,120,15000,Morgan Patel,Email Open,No,Downloaded Whitepaper,Anthony White,True,Assigned,Morgan Patel


In [12]:
sales_table_df = df.groupby('round_robin_assignment').agg(
    leads = ('lead_id', 'count')
)

sales_table_df

,leads
round_robin_assignment,
Alex Kim,31
Chris Brown,24
Jordan Lee,20
Morgan Patel,24
Taylor Smith,24


In [13]:
df['last_activity'].unique()


array(['Phone Call', 'Form Submission', 'Demo Requested', 'Email Open',
       'No Activity', 'Content Download', 'Website Visit'], dtype=object)

In [39]:
dead_leads = ['Converted', 'Unqualified', 'Lost']
df= df[~df['lead_status'].isin(dead_leads)]

df['lead_status'].unique()

array(['Nurturing', 'New', 'Contacted', 'Qualified'], dtype=object)

In [40]:
def calculate_lead_score(last_activity, lead_score, job_title, lead_status, notes, deal_value):
    
    score = 0
    
    # -------------------------
    # Intent / Last Activity
    # -------------------------
    activity = str(last_activity).lower()
    
    if 'Demo' in activity:
        score += 20
    elif 'Contact' in activity:
        score += 18    
    elif 'Phone' in activity:
        score += 15
    elif 'Form' in activity:
        score += 12
    elif 'Content' in activity:
        score += 10
    elif 'Email' in activity:
        score += 8
    elif 'Website' in activity or 'web' in activity:
        score += 5
    
    
    # -------------------------
    # Lead Score
    # -------------------------
    if lead_score >= 90:
        score += 20
    elif lead_score >= 80:
        score += 15
    elif lead_score >= 70:
        score += 10
    elif lead_score >= 60:
        score += 8
    elif lead_score >= 50:
        score += 5
    elif lead_score >= 40:
        score += 2
    
    
    # -------------------------
    # Job Title / Seniority
    # -------------------------
    title = str(job_title).lower()
    
    if any(x in title for x in [
        'ceo', 'cmo', 'cto', 'cio', 
        'chief', 'founder', 'owner'
    ]):
        score += 20
        
    elif any(x in title for x in [
        'vp', 'vice president'
    ]):
        score += 18
        
    elif any(x in title for x in [
        'director', 'head'
    ]):
        score += 15
        
    elif any(x in title for x in [
        'senior manager', 'senior'
    ]):
        score += 12
        
    elif 'manager' in title:
        score += 10
        
    elif any(x in title for x in [
        'lead', 'principal'
    ]):
        score += 8
    
    
    
    status = str(lead_status).lower()
    if any(x in status for x in [
        'lost', 'unqualified', 'converted'
    ]):
        score += 0
    elif any(x in status for x in [
        'qualified', 'new'
    ]): 
        score += 15
    elif any(x in status for x in [
        'contacted'
    ]):
        score += 10 
    elif any(x in status for x in [
        'nurturing'
    ]): 
        score += 5

    notes_text = str(notes).lower()
    if any(x in notes_text for x in [
        'demo', 'pricing'
    ]):
        score += 15
    elif any(x in notes_text for x in [
        'follow', 'referral'
    ]):
        score += 10
    elif any(x in notes_text for x in [
        'interested', 'downloaded'
    ]):
        score += 5
    elif any(x in notes_text for x in [
        'wrong', 'no response', 'duplicate'
    ]):
        score += 0
    
    # -------------------------
    # Estimated Deal Value
    # -------------------------
    if deal_value >= 100000:
        score += 15
    elif deal_value >= 75000:
        score += 12
    elif deal_value >= 50000:
        score += 10
    elif deal_value >= 25000:
        score += 7
    elif deal_value >= 10000:
        score += 5
    
    
    return score

df['lead_priority_score'] = df.apply(
    lambda row: calculate_lead_score(
        row['last_activity'],
        row['lead_score'],
        row['job_title'],
        row['lead_status'],
        row['notes'],
        row['estimated_deal_value_usd']
    ),
    axis=1
)

df['score_percentile'] = df['lead_priority_score'].rank(
    pct=True,
    method='average'
)

df['priority'] = pd.cut(
    df['score_percentile'],
    bins=[0, .10, .20, .30, .40, .50, .60, .70, .80, .90, 1.00],
    labels=[10, 9, 8, 7, 6, 5, 4, 3, 2, 1],
    include_lowest=True
).astype(int)
    
df.head()

/var/folders/12/6m8mdrf16tnfb04x574t07r80000gn/T/ipykernel_30734/79849841.py:132: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/12/6m8mdrf16tnfb04x574t07r80000gn/T/ipykernel_30734/79849841.py:144: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/12/6m8mdrf16tnfb04x574t07r80000gn/T/ipykernel_30734/79849841.py:149: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

,lead_id,lead_first_name,lead_last_name,job_title,company_name,industry,company_size,city,state,country,...,last_activity,marketing_opt_in,notes,lead_full_name,is_decision_maker,is_assigned,round_robin_assignment,lead_priority_score,priority,score_percentile
0,L-10019,James,Moore,Founder,Silverstone Automotive,Automotive,501-1000,Columbus,OH,United Sates,...,Phone Call,NaN,Downloaded Whitepaper,James Moore,True,Assigned,Alex Kim,37,9,0.177632
1,L-10046,Andrew,Jackson,Sales Director,Lighthouse Biotech,Biotechnology,1-50,Cambridge,MA,United Sates,...,Form Submission,NaN,Asked For Pricing,Andrew Jackson,True,Assigned,Chris Brown,77,1,1.000000
2,L-10048,Christopher,Williams,Product Marketing Manager,Vertex Software,Technology,5000+,Austin,TX,United Sates,...,Form Submission,NaN,Referral From Partner,Christopher Williams,False,Unassigned,Alex Kim,47,5,0.578947
3,L-10090,Andrew,Brown,Data Analytics Manager,Westbridge Legal,Legal,1001-5000,Denver,CO,United Sates,...,Demo Requested,Yes,Duplicate Suspected,Andrew Brown,False,Assigned,Jordan Lee,42,6,0.447368
4,L-10005,Emily,Garcia,Revops Manager,Metro Hospitality,Hospitality,5000+,Miami,FL,United Sates,...,Phone Call,No,Referral From Partner,Emily Garcia,False,Assigned,Chris Brown,30,10,0.046053


In [41]:
columns = ['priority', 'lead_full_name', 'job_title', 'company_name', 'industry', 'phone', 'email', 'lead_status', 'lead_source', 'last_activity', 'estimated_deal_value_usd', 'lead_score', 'round_robin_assignment','notes']

priority_view = (
    df[columns].loc[df['lead_status'] != 'Lost'].sort_values('priority', ascending = True)
)

priority_view.columns = (
    priority_view.columns.str.replace('_', ' ', regex=False).str.title().str.replace('id', 'ID', regex = False).str.replace('Usd', 'USD', regex = False)
)

priority_view.head(10)

,Priority,Lead Full Name,Job Title,Company Name,Industry,Phone,Email,Lead Status,Lead Source,Last Activity,Estimated Deal Value USD,Lead Score,Round Robin Assignment,Notes
57,1,Jennifer Perez,Director Of Marketing,Pioneer Logistics,Logistics,(935) 551-4335,jennifer.perez@pioneerlogistics.com,New,Website,No Activity,5000,91,Alex Kim,Referral From Partner
1,1,Andrew Jackson,Sales Director,Lighthouse Biotech,Biotechnology,(355) 650-2118,andrew.jackson@lighthousebiotech.com,New,Google Ads,Form Submission,40000,97,Chris Brown,Asked For Pricing
83,1,Nicole Moore,Director Of Marketing,Cedar Manufacturing,Manufacturing,(395) 981-3610,nicole.moore@cedarmanufacturing.com,New,Website,Form Submission,40000,70,Alex Kim,Requested Demo
74,1,Anthony Anderson,Director Of Marketing,Blue Ridge Financial,Financial Services,(276) 750-4492,anthony.anderson@blueridgefinancial.com,Qualified,Organic Search,Website Visit,75000,63,Alex Kim,Interested In Enterprise Plan
64,1,David Lee,Vp Marketing,Acme Health Systems,Healthcare,(597) 782-7895,david.lee@acmehealthsystems.com,New,Website,Email Open,75000,69,Morgan Patel,Referral From Partner
59,1,Emily Smith,Vp Marketing,Clearwater Foods,Consumer Goods,(829) 406-3137,emily.smith@clearwaterfoods.com,Qualified,Website,Demo Requested,15000,85,Taylor Smith,Follow Up Next Week
11,1,Robert Miller,Sales Director,Summit Retail Group,Retail,(607) 445-6269,robert.miller@summitretailgroup.com,Contacted,Google Ads,Email Open,75000,92,Jordan Lee,Referral From Partner
35,1,Jennifer Perez,Director Of Marketing,Pioneer Logistics,Logistics,(935) 551-4335,jennifer.perez@pioneerlogistics.com,New,Website,No Activity,5000,91,Alex Kim,Referral From Partner
106,1,Robert Miller,Vp Marketing,Brightpath Education,Education,(378) 765-2229,robert.miller@brightpatheducatio.com,Contacted,Linkedin,Website Visit,5000,96,Chris Brown,Follow Up Next Week
28,1,Jennifer Taylor,Vp Marketing,Westbridge Legal,Legal,(483) 394-3001,jennifer.taylor@westbridgelegal.com,New,Paid Search,Form Submission,100000,22,Jordan Lee,Asked For Pricing
